# Day 38 — Tree-based models: Decision Trees & Random Forests
Objectives:
- Train tree and RF.
- Feature importances.
- Overfitting control (max_depth, min_samples_*).


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
X,y = load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y,random_state=42)
dt = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xtr,ytr)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr,ytr)
dt.score(Xte,yte), rf.score(Xte,yte)


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — tree splits, ensemble variance reduction, and held-out importance

### Mental model

A decision tree recursively partitions feature space. Each split asks a
threshold question chosen to reduce impurity; each leaf stores a local
prediction. Deep trees can represent fine interactions but also isolate
noise. Depth, leaf size, and pruning control that capacity.

A random forest trains many trees on bootstrapped rows while considering
subsets of features. Averaging decorrelated trees reduces variance.
Impurity importance describes split usage inside the fitted forest;
held-out permutation importance measures score loss when one feature's
association is broken. Neither establishes causality.

### Read the API before running it

- **`DecisionTreeClassifier(max_depth=..., min_samples_leaf=...)`:** sets capacity controls before fitting and exposes tree-specific train/validation gaps.
- **`RandomForestClassifier(n_estimators=..., random_state=...)`:** averages bootstrapped, feature-subsampled trees; enough estimators stabilize rather than deepen the model.
- **`permutation_importance(model, X_valid, y_valid, ...)`:** measures held-out score change under repeated feature shuffles.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — make the depth-versus-generalization gap visible

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** The validation split represents future cases and was not used to choose unlimited alternatives.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, stratify=y, random_state=3801
)
for depth in (1, 3, 8, None):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=3801)
    tree.fit(X_train, y_train)
    print(depth, {"train": tree.score(X_train, y_train),
                  "valid": tree.score(X_valid, y_valid)})

**Expected observation:** Training accuracy rises with capacity; validation accuracy need not, revealing overfitting rather than a syntax error.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — test a deliberately useless noise feature

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** The model's held-out score is good enough that score perturbations are interpretable.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
rng = np.random.default_rng(3802)
X = np.column_stack([X, rng.normal(size=X.shape[0])])
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, stratify=y, random_state=3802
)
forest = RandomForestClassifier(n_estimators=80, random_state=3802, n_jobs=1)
forest.fit(X_train, y_train)
importance = permutation_importance(
    forest, X_valid, y_valid, n_repeats=5, random_state=3802, n_jobs=1
)
print({"noise_importance": importance.importances_mean[-1],
       "best_importance": importance.importances_mean.max()})

**Expected observation:** The random noise feature should have importance near zero, while at least one real feature matters more.

### Debugging and practice ramp

**Common mistake:** Treating `feature_importances_` as causal effect or as reliable when correlated features can substitute for one another.

**Diagnostic:** Compare train/validation scores, tree depth/leaf counts, repeated permutation intervals, and a synthetic noise feature.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define tree splits, ensemble variance reduction, and held-out importance in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not interpret importance from a poorly performing model or from the same rows used to fit it.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Plot tree depth versus accuracy.

**Verify:** For task `Plot tree depth versus accuracy`, show the labeled figure and reconcile it with a numeric summary so appearance is not the only check; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






2. Inspect feature importances and discuss their reliability.

**Verify:** For task `Inspect feature importances and discuss their reliability`, demonstrate the concrete requirement “2. Inspect feature importances and discuss their reliability” with explicit inputs, observable output, and one counterexample.







### Progressive hints

1. Record training and validation accuracy for each depth. Treat `None` as an
   unbounded depth label rather than a numeric x-coordinate.
2. Compare `feature_importances_` with
   `sklearn.inspection.permutation_importance` on held-out data. Preserve feature
   names from `load_breast_cancer().feature_names`.

### Additional mastery practice

Diagnose tree capacity with train/validation evidence and inspect importance with methods that respect held-out data, correlation, and class costs.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

3. **Pruning implementation:** Use a decision tree's cost-complexity pruning path to evaluate candidate `ccp_alpha` values with cross-validation. Freeze the chosen value before final holdout evaluation.
   **Progressive hint:** The path is derived from training data. Treat alpha selection as a hyperparameter search inside the development boundary.

**Verify:** For task `Pruning implementation: Use a decision tree's cost-complexity pruning path to evaluate candid...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.







4. **Out-of-bag reasoning:** Enable `oob_score=True` in a RandomForestClassifier and compare the out-of-bag estimate with held-out or cross-validated performance.
   **Progressive hint:** Each tree leaves out about 36.8% of bootstrap rows; aggregate predictions only from trees for which a row was out of bag.

**Verify:** For task `Out-of-bag reasoning: Enable oobscore=True in a RandomForestClassifier and compare the out-of...`, record the seed, resampling unit, run count, estimate, and an analytic or hand-worked comparison with a stated tolerance; then assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior.







5. **Imbalance debugging:** Train a tree on a 98:2 dataset, compare accuracy with minority recall and average precision, then test `class_weight='balanced'`.
   **Progressive hint:** A majority-only classifier reaches 98% accuracy. Keep the split stratified and compare confusion matrices at a documented threshold.

**Verify:** For task `Imbalance debugging: Train a tree on a 98:2 dataset, compare accuracy with minority recall an...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.







6. **Correlated-importance edge case:** Duplicate one informative feature, refit the forest, and observe how impurity and single-feature permutation importance change.
   **Progressive hint:** The two columns can substitute for each other, splitting apparent importance and making either single-column permutation look weak.

**Verify:** For task `Correlated-importance edge case: Duplicate one informative feature, refit the forest, and obs...`, record the seed, resampling unit, run count, estimate, and an analytic or hand-worked comparison with a stated tolerance; then report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 3 — Pruning implementation


# Practice 4 — Out-of-bag reasoning


# Practice 5 — Imbalance debugging


# Practice 6 — Correlated-importance edge case
